In [ ]:
import json
import random
import geopandas as gpd
from shapely.geometry import Point

# ---- Number of points ----
N = 400

shp_path = "ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp"
world = gpd.read_file(shp_path)

# ---- Select USA ----
us = world[world["ADMIN"] == "United States of America"].to_crs("EPSG:4326")

# ---- Get US bounds ----
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = us.total_bounds

# ---- Generate points over land ----
points = []
attempts = 0
while len(points) < N:
    lat = random.uniform(LAT_MIN, LAT_MAX)
    lon = random.uniform(LON_MIN, LON_MAX)
    p = Point(lon, lat)
    if us.contains(p).any():
        points.append({"lat": round(lat, 6), "lon": round(lon, 6)})
    attempts += 1
    if attempts > 5000:
        raise RuntimeError("Too many attempts to generate points over land. Try again.")

# ---- Save to JSON ----
output_file = f"us-samples/random_coords_{N}.json"
with open(output_file, "w") as f:
    json.dump(points, f, indent=2)

print(f"✔ Generated {N} random coordinates over US land and saved to {output_file}")
print(json.dumps(points, indent=2))